# Selective Classification with Combined OOD and Model Uncertainty Estimation

The following demonstrates how the selective classification performance with combined OOD and model uncertainty estimation was evaluated. We assume model training and evaluation has already been run (see notebook 01) and use the sampling rate perturbation at strength 200 Hz as example.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from math import ceil
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.isotonic import IsotonicRegression

from oddeeg.utils import construct_results_path
from oddeeg.aggregators import CalibratedMaxQuantile

In [2]:
perturbation = "perturbation_sfreq"
strength = "200"

dsc_train_cfg = {
    "dataset_name": "TUAB",
    "task": "normality",
    "model_name": "TCN",
    "training_mode": "discriminative"
}

gen_train_cfg = {
    "dataset_name": "TUAB",
    "task": None,  # unconditional generative model
    "model_name": "UNet",
    "training_mode": "flow_matching",
    "max_batches": 1000000,
}

In [3]:
PERTURBATION_META = {
    "perturbation_sfreq": {"id_sentinel": 100},
    "perturbation_channel_shuffle": {"id_sentinel": 0},
    "perturbation_highpass_hz": {"id_sentinel": 0.0},
    "perturbation_lowpass_hz": {"id_sentinel": float("inf")},
    "perturbation_reref_scheme": {"id_sentinel": "average"},
}

In [ ]:
# Fit OOD detection methods using in-distribution validation data
preds_id_val_gen = pd.read_csv(construct_results_path(config=gen_train_cfg, split_pick="val"))
preds_id_val_dsc = pd.read_csv(construct_results_path(config=dsc_train_cfg, split_pick="val"))
preds_id_val = preds_id_val_gen.merge(preds_id_val_dsc)

# Note that we are using CalibratedMaxQuantile here to get calibrated SITN scores
sitn_calibrated = CalibratedMaxQuantile({"anderson_darling_statistic": True, "ps_cv": True})
sitn_calibrated.fit(preds_id_val)

# Calibrate MSP model uncertainty estimates using isotonic regression
val_sorted = preds_id_val.sort_values("msp")
msp = val_sorted["msp"].values
is_correct = (val_sorted["pred"] == val_sorted["target"]).astype(float).values

iso_reg = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip")
iso_acc = iso_reg.fit_transform(msp, is_correct)

target_acc = 0.6
msp_threshold = val_sorted["msp"].values[iso_acc >= target_acc].min()
print(f"MSP Threshold for a target accuracy of {target_acc * 100}%: {msp_threshold}")

In [5]:
id_sentinel = PERTURBATION_META.get(perturbation, {}).get("id_sentinel", 0)

# Load ID test preds
preds_id_test_gen = pd.read_csv(construct_results_path(config=gen_train_cfg, split_pick="test"))
preds_id_test_dsc = pd.read_csv(construct_results_path(config=dsc_train_cfg, split_pick="test"))
preds_id = preds_id_test_gen.merge(preds_id_test_dsc)
preds_id[perturbation] = id_sentinel

# Load OOD test preds
results_path = construct_results_path(config=gen_train_cfg, split_pick="test", **{perturbation: strength})
if not results_path.exists():
    raise FileNotFoundError(f"Results not found: {results_path}")
preds_ood_gen = pd.read_csv(results_path)

results_path = construct_results_path(config=dsc_train_cfg, split_pick="test", **{perturbation: strength})
if not results_path.exists():
    raise FileNotFoundError(f"Results not found: {results_path}")
preds_ood_dsc = pd.read_csv(results_path)

preds_ood = preds_ood_gen.merge(preds_ood_dsc)
preds_ood[perturbation] = strength

# Combine ID and OOD preds
preds = pd.concat([preds_id, preds_ood], ignore_index=True)
preds["correct"] = preds["target"] == preds["pred"]
preds["ood"] = preds[perturbation] != id_sentinel

# Add calibrated SITN scores
preds["sitn_calibrated"] = sitn_calibrated.score(preds)

# Update ID and OOD preds after adding new metrics
preds_id = preds[~preds["ood"]]
preds_ood = preds[preds["ood"]]

In [6]:
def evaluate_rejection(df, reject_mask, name):
    retained_df = df[~reject_mask]
    coverage = len(retained_df) / len(df)
    
    if len(retained_df) > 0:
        accuracy = (retained_df["pred"] == retained_df["target"]).mean()
    else:
        accuracy = 0.0
        
    rejected_correct = (df[reject_mask]["pred"] == df[reject_mask]["target"]).sum() if reject_mask.sum() > 0 else 0
    print(f"{name:<22} | Coverage: {coverage*100:5.1f}% | Retained Acc: {accuracy*100:5.1f}%")

#### Selective Classification on ID Data Only

In [ ]:
is_ood = preds_id["sitn_calibrated"] >= 0.99
is_uncertain = preds_id["msp"] < msp_threshold
combined_rejection = is_ood | is_uncertain

evaluate_rejection(preds_id, np.zeros(len(preds_id), dtype=bool), "Baseline (No Rejection)")
evaluate_rejection(preds_id, is_ood, "OOD Detection (SITN) Only")
evaluate_rejection(preds_id, is_uncertain, "Model Uncertainty (MSP) Only")
evaluate_rejection(preds_id, combined_rejection, "OOD and Model Uncertainty Combined (SITN+MSP)")

#### Selective Classification on Pooled ID and OOD Data

In [ ]:
is_ood = preds["sitn_calibrated"] >= 0.99
is_uncertain = preds["msp"] < msp_threshold
combined_rejection = is_ood | is_uncertain

evaluate_rejection(preds, np.zeros(len(preds), dtype=bool), "Baseline (No Rejection)")
evaluate_rejection(preds, is_ood, "OOD Detection (SITN) Only")
evaluate_rejection(preds, is_uncertain, "Model Uncertainty (MSP) Only")
evaluate_rejection(preds, combined_rejection, "OOD and Model Uncertainty Combined (SITN+MSP)")